In [1]:
from datasets import load_from_disk, load_dataset
from PIL import Image
import random

/home/jinaai/miniconda3/envs/rerankvlm_datagen_cpi_1_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
output_dir = "/data/hf_datasets/ocr_t2s_train_set_colpali_jina3_train5000_4096/"#rerank_train_set_colpali_1_2merge_test50 #/data/hf_datasets/ocr_erank_train_set_colpali_1_2merge_test500_multi/part_1
loaded_dataset = load_from_disk(output_dir)
print("loaded_dataset",load_dataset)
# loaded_dataset = load_dataset(output_dir, split="train[:50]")
print(f"Loaded dataset sample: {loaded_dataset[0]}")
print(f"Number of samples loaded: {len(loaded_dataset)}")

loaded_dataset <function load_dataset at 0x7f38e0353100>
Loaded dataset sample: {'query': 'Comparing panels a, b, c, and d, which statement best describes the data variance?', 'positive_index': 0, 'topk_indices': [4475, 4919, 1646, 2860, 250, 382, 1715, 4319, 1655, 1375, 820, 4406, 2953, 348, 4708, 234, 1714, 2139, 4554, 4399, 3114, 4246, 3892, 2665, 2209, 1830], 'negative_indices': [4475, 4919, 1646, 2860, 250, 382, 1715, 4319, 1655, 1375, 820, 4406, 2953, 348, 4708, 234, 1714, 2139, 4554, 4399, 3114, 4246, 3892, 2665, 2209, 1830], 'scores': [0.6850959658622742, 0.6802697777748108, 0.667136549949646, 0.6667993664741516, 0.6654625535011292, 0.6648480296134949, 0.6645611524581909, 0.6636054515838623, 0.6627421975135803, 0.6615692377090454, 0.6596399545669556, 0.6594265103340149, 0.6579838991165161, 0.6562997102737427, 0.6551237106323242, 0.6524257659912109, 0.652172327041626, 0.6521653532981873, 0.6515663266181946, 0.6508280038833618, 0.6473879218101501, 0.6454125642776489, 0.6444950103

In [15]:
import numpy as np
def dcg_at_k(relevance, k=5, method=0):
    """
    计算DCG@k
    :param relevance: 相关性分数列表
    :param k: 返回的项数
    :param method: 计算方法，0或1
    :return: DCG@k的值
    """
    if method == 0:
        gain = [pow(2, rel) - 1.0 for rel in relevance[:k]]
    elif method == 1:
        gain = [pow(2, rel) / np.log2(i + 2) for i, rel in enumerate(relevance[:k])]
    else:
        assert False, 'invalid method'
    
    return np.sum(gain)

def ndcg_at_k(relevance, scores, k=5):
    """
    计算NDCG@k
    :param relevance: 相关性分数列表
    :param scores: 模型给出的分数列表
    :param k: 返回的项数
    :return: NDCG@k的值
    """
    # 计算DCG@k
    dcg_max = dcg_at_k(sorted(relevance, reverse=True), k)
    # 计算实际的DCG@k
    actual_dcg = dcg_at_k(scores, k)
    
    # 计算NDCG@k
    ndcg = actual_dcg / dcg_max
    return ndcg

In [30]:
b = 100
queries = loaded_dataset[:b]
# print(queries['scores'])
for i in range(6):
    random.seed()
    seed = random.randint(0, b-1)
    
    print(f"Seed: {seed}")
    relevance = [1.0] 
    scores = queries['scores'][seed]
    k = min(len(scores), 10)  
    ndcg = ndcg_at_k(relevance, scores, k)
    print(f"Query: {queries['query'][seed]}")
    print(f"NDCG@{k}: {ndcg}")
    print("\n")

Seed: 94
Query: What is the dotted line in the graph?
Your answer should be compact.
NDCG@10: 6.4562442440302465


Seed: 3
Query: What is the date?
Your answer should be very brief.
NDCG@10: 6.453254514953228


Seed: 41
Query: Who sponsores Community Conference on Food and Population?
Keep it short and to the point.
NDCG@10: 6.718170516659079


Seed: 78
Query: What is the main challenge facing energy companies according to the introduction?
NDCG@10: 5.909878605249308


Seed: 22
Query: What is the nature of the form?
Ensure brevity in your answer. 
NDCG@10: 5.954739088808808


Seed: 16
Query: What trend can be observed in figure a when the strain (ε) is increased from 0% to 8%?
NDCG@10: 6.239572055054932




In [4]:
from PIL import Image

# img_list = loaded_dataset['Positive_image']
img0 = loaded_dataset[0]['image']
# img41 = loaded_dataset[11]['positive_image']

In [ ]:
seed = 0
print(f"Query: {loaded_dataset[seed]['query']}")
print(f"text: {loaded_dataset[seed]['text']}")
print(f"idx: {loaded_dataset[seed]['raw_id']}")


In [ ]:
img0

In [4]:
for i in range(6):
    random.seed()
    seed = random.randint(0, len(loaded_dataset)-1)
    print(f"Query: {loaded_dataset[seed]['query']}")
    # print(f"Positive Image: {loaded_dataset[seed]['positive_image']}")
    print(f"Positive Index: {loaded_dataset[seed]['positive_index']}")
    print(f"top_k Indices: {loaded_dataset[seed]['topk_indices']}")#negative_indices
    print(f"Scores: {loaded_dataset[seed]['scores']}")
    print("\n")


Query: What does the function animateComputerMoving do?
Positive Index: 877
top_k Indices: [877, 2307, 1528, 1646, 3074, 967, 4961, 572, 2566, 744, 2432, 2675, 298, 1461, 4482, 2420, 288, 2162, 4094, 4353, 4255, 3090, 3560, 2638, 4032, 2774]
Scores: [0.7172300815582275, 0.6409473419189453, 0.6395168304443359, 0.6331759691238403, 0.6331455111503601, 0.6257294416427612, 0.6155081987380981, 0.6135450601577759, 0.6135210394859314, 0.6120305061340332, 0.6120114326477051, 0.6117995977401733, 0.6111561059951782, 0.6100271940231323, 0.6094464063644409, 0.6090855598449707, 0.6082816123962402, 0.6076807975769043, 0.6055832505226135, 0.6054654717445374, 0.6032021045684814, 0.6029560565948486, 0.6029016375541687, 0.6016894578933716, 0.5992536544799805, 0.5992536544799805]


Query: What is the amount given for 1974 intra-science conference on new ideas in cancer chemotherapy ?
Make the answer very short.
Positive Index: 3533
top_k Indices: [3533, 313, 995, 3282, 4452, 3276, 1455, 868, 2318, 2629, 3

In [ ]:
loaded_dataset[31]['query']

In [ ]:
loaded_dataset[31]['image']

In [ ]:
loaded_dataset[41]['image']

In [ ]:
loaded_dataset[7]['image']

In [ ]:
loaded_dataset[7]['query']

In [ ]:
img4

In [ ]:
img41

In [ ]:
loaded_dataset[1]['positive_image']

In [ ]:
loaded_dataset[5]['positive_image']